# DOMINANT — step by step, on a real dataset

Reproducing Ding, Li, Bhanushali, Liu, *"Deep Anomaly Detection on Attributed Networks"* (SDM 2019).

This notebook builds DOMINANT manually with PyTorch + PyTorch Geometric — every step is visible in a separate cell.

## Step 1 — Install dependencies

Colab already has PyTorch. We just need PyTorch Geometric and PyGOD (PyGOD gives us the benchmark dataset with anomalies already injected, so we don't have to write the injection code ourselves).

In [ ]:
!pip install -q torch_geometric pygod


## Step 2 — Load a real graph with real injected anomalies

`inj_cora` is the Cora citation network (papers = nodes, citations = edges, bag-of-words = attributes) with the *same style* of clique + attribute-swap anomalies you saw injected in the from-scratch NumPy version — except this time it's a standard, citable benchmark dataset used across the GAD literature (via the BOND benchmark).

In [ ]:
import torch
from pygod.utils import load_data
import torch_geometric # Import torch_geometric to access its storage module
from torch.serialization import add_safe_globals

# Add the required global to the safe globals list for torch.load.
# This is necessary because PyTorch 2.6 changed `weights_only` default to True
# and `pygod` internally calls `torch.load` which tries to load this global.
add_safe_globals([torch_geometric.data.storage.GlobalStorage])

data = load_data('inj_cora')
print(f"Nodes: {data.num_nodes}")
print(f"Edges: {data.num_edges}")
print(f"Attribute dimensions: {data.x.shape[1]}")
print(f"Anomalies: {int(data.y.sum())} ({100*data.y.float().mean():.1f}%)")

Nodes: 2708
Edges: 11060
Attribute dimensions: 1433
Anomalies: 210 (7.8%)


## Step 3 — The graph normalization (handled for us)

In the from-scratch version we hand-built `D^-1/2 (A+I) D^-1/2` ourselves. Here, PyTorch Geometric's `GCNConv` layer does that normalization internally every time it's called — same math, just wrapped in a tested library function. Nothing to run in this cell; just know it's happening inside every `GCNConv` call below.

## Step 4 — Build the encoder: 3 stacked "blend with neighbors" layers

Each `GCNConv` layer = one round of every node averaging in its neighbors' values, then a small learned transformation. Stack 3 of them, and information from 3 hops away reaches every node.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Encoder(nn.Module):
    def __init__(self, in_dim, h1=64, h2=32, h3=16):
        super().__init__()
        self.conv1 = GCNConv(in_dim, h1)
        self.conv2 = GCNConv(h1, h2)
        self.conv3 = GCNConv(h2, h3)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        z = self.conv3(h, edge_index)  # no activation on the final embedding
        return z  # this is Z, the embedding for every node


## Step 5 — Build the two decoders

**Structure decoder:** compare every pair of node embeddings — similar embeddings imply a guessed connection. **Attribute decoder:** one more "blend with neighbors" layer, trying to regenerate the original attribute values from Z.

In [ ]:
class AttributeDecoder(nn.Module):
    def __init__(self, embed_dim, out_dim):
        super().__init__()
        self.conv = GCNConv(embed_dim, out_dim)

    def forward(self, z, edge_index):
        return F.relu(self.conv(z, edge_index))

def structure_decoder(z):
    return z @ z.t()  # raw logits now, not a probability yet

class Dominant(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.encoder = Encoder(in_dim)
        self.attr_decoder = AttributeDecoder(embed_dim=16, out_dim=in_dim)

    def forward(self, x, edge_index):
        z = self.encoder(x, edge_index)
        s_logits = structure_decoder(z)
        x_hat = self.attr_decoder(z, edge_index)
        return s_logits, x_hat


## Step 6 — Loss function: how wrong were both guesses?

`alpha` balances structure error vs attribute error — the paper finds the sweet spot is usually between 0.4 and 0.8, not at either extreme. We'll build the real (dense) adjacency matrix once, since we need to compare against it directly.

In [ ]:
from torch_geometric.utils import to_dense_adj

num_nodes = A.shape[0]
num_edges = A.sum()
pos_weight = torch.tensor((num_nodes * num_nodes - num_edges) / num_edges)
norm = num_nodes * num_nodes / float((num_nodes * num_nodes - num_edges) * 2)

def loss_fn(A, X, s_logits, X_hat, alpha=0.6):
    struct_err = norm * F.binary_cross_entropy_with_logits(s_logits, A, pos_weight=pos_weight)
    attr_err = F.mse_loss(X_hat, X)
    return (1 - alpha) * struct_err + alpha * attr_err


/tmp/ipykernel_2267/3228408397.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight = torch.tensor((num_nodes * num_nodes - num_edges) / num_edges)


## Step 7 — Train: repeatedly nudge the weights to reduce the loss

This is the only genuinely iterative part. PyTorch's autograd computes all the gradients for us here (unlike the from-scratch NumPy version, where we derived and verified them by hand).

In [ ]:
model = Dominant(in_dim=X.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
alpha = 0.6
n_epochs = 300

for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    s_logits, X_hat = model(X, data.edge_index)
    loss = loss_fn(A, X, s_logits, X_hat, alpha)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0 or epoch == 1:
        print(f"epoch {epoch:4d}  loss={loss.item():.6f}")


epoch    1  loss=0.277283
epoch   50  loss=0.203894
epoch  100  loss=0.194397
epoch  150  loss=0.183324
epoch  200  loss=0.176623
epoch  250  loss=0.171746
epoch  300  loss=0.169118


## Step 8 — Score and rank every node

Per-node anomaly score = weighted combination of how badly that node's row reconstructed, for both structure and attributes.

In [ ]:
model.eval()
with torch.no_grad():
    s_logits, X_hat = model(X, data.edge_index)
    A_hat = torch.sigmoid(s_logits)
    struct_err = torch.norm(A - A_hat, dim=1)
    attr_err = torch.norm(X - X_hat, dim=1)
    scores = ((1 - alpha) * struct_err + alpha * attr_err).numpy()

labels = data.y.numpy()


## Step 9 — Evaluate: did the top of the ranking match the real anomalies?

Compare against the paper's own reported range (~0.75–0.82 ROC-AUC on BlogCatalog/Flickr/ACM) — `inj_cora` is a different dataset so an exact match isn't expected, but it should land in a similar ballpark, since it's the same method and a similar injection process.

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np # Import numpy to use np.unique


binary_labels = (labels != 0).astype(int)  # any injected anomaly type counts


unique_binary_labels = np.unique(binary_labels)
if len(unique_binary_labels) < 2:
    print("Cannot compute ROC-AUC: 'binary_labels' array contains less than two unique classes (e.g., all 0s or all 1s after binarization).")
    auc = np.nan # Set AUC to NaN if it cannot be computed
else:
    auc = roc_auc_score(binary_labels, scores) # Use binary_labels here
    print(f"ROC-AUC: {auc:.4f}")

order = scores.argsort()[::-1]

num_true_anomalies = binary_labels.sum()

for k in (50, 100, 200):
    top_k_labels = binary_labels[order[:k]] # Use binary_labels here
    num_anomalies_in_top_k = top_k_labels.sum()

    precision = num_anomalies_in_top_k / k

    if num_true_anomalies > 0:
        recall = num_anomalies_in_top_k / num_true_anomalies
    else:
        recall = np.nan # Set recall to NaN if no true anomalies exist

    print(f"Precision@{k}: {precision:.3f}   Recall@{k}: {recall:.3f}")

ROC-AUC: 0.7826
Precision@50: 0.180   Recall@50: 0.065
Precision@100: 0.280   Recall@100: 0.203
Precision@200: 0.220   Recall@200: 0.319


## Bonus — sweep alpha and reproduce the paper's Figure 3

Retrain quickly across a few alpha values and check whether the best AUC lands in the 0.4–0.8 range, same as the paper's own finding.

In [ ]:
results = {}

binary_labels = (labels != 0).astype(int)  # any injected anomaly type counts


unique_binary_labels = np.unique(binary_labels)
if len(unique_binary_labels) < 2:
    print("Cannot compute ROC-AUC: 'binary_labels' array contains less than two unique classes (e.g., all 0s or all 1s after binarization).")

else:
    for a in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
        m = Dominant(in_dim=X.shape[1])
        opt = torch.optim.Adam(m.parameters(), lr=0.005)
        for epoch in range(150):
            opt.zero_grad()
            A_hat, X_hat = m(X, data.edge_index)
            l = loss_fn(A, X, A_hat, X_hat, alpha=a)
            l.backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            A_hat, X_hat = m(X, data.edge_index)
            s = ((1 - a) * torch.norm(A - A_hat, dim=1) + a * torch.norm(X - X_hat, dim=1)).numpy()

        # Use binary_labels for roc_auc_score
        results[a] = roc_auc_score(binary_labels, s)
        print(f"alpha={a:.1f}  AUC={results[a]:.4f}")

    best_alpha = max(results, key=results.get)
    print(f"\nBest alpha: {best_alpha} (AUC={results[best_alpha]:.4f})")


alpha=0.0  AUC=0.6782
alpha=0.2  AUC=0.6874
alpha=0.4  AUC=0.7008
alpha=0.6  AUC=0.6750
alpha=0.8  AUC=0.6774
alpha=1.0  AUC=0.7272

Best alpha: 1.0 (AUC=0.7272)


In [ ]:
print(f"Mean of true A: {A.mean().item():.5f}")
print(f"Mean of A_hat:  {A_hat.mean().item():.5f}")
print(f"Fraction of true edges: {(A.sum() / A.numel()).item():.5f}")

Mean of true A: 0.00151
Mean of A_hat:  0.01544
Fraction of true edges: 0.00151


In [ ]:
print(np.unique(data.y, return_counts=True))

(array([0, 1, 2, 3]), array([2570,   68,   68,    2]))


In [ ]:
print(f"Mean struct_err (normal):  {struct_err[labels==0].mean():.4f}")
print(f"Mean struct_err (anomaly): {struct_err[labels==1].mean():.4f}")
print(f"Mean attr_err (normal):    {attr_err[labels==0].mean():.4f}")
print(f"Mean attr_err (anomaly):   {attr_err[labels==1].mean():.4f}")

Mean struct_err (normal):  27.0709
Mean struct_err (anomaly): 27.0615
Mean attr_err (normal):    0.2478
Mean attr_err (anomaly):   0.4047
